# 01 · Data cleaning & merging
Combines the Kaggle food-delivery orders with Open-Meteo weather.

All logic lives in `src/merge_weather.py` and `src/clean_orders.py`; this notebook runs it step by step and shows the data **before and after** each fix.

In [1]:
import sys, pandas as pd
sys.path.insert(0, '../src')
import merge_weather as mw, clean_orders as co
pd.set_option('display.max_columns', 40)

## 1. Raw orders – what is wrong

In [2]:
raw = pd.read_csv(co.RAW)
print(raw.shape)
raw.head(3)

(45593, 20)


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26


In [3]:
# trailing spaces, text 'NaN', prefixes in values
for c in ['Weatherconditions', 'Road_traffic_density', 'City', 'Time_taken(min)']:
    print(c, raw[c].unique()[:6])

Weatherconditions <ArrowStringArray>
[     'conditions Sunny',     'conditions Stormy', 'conditions Sandstorms',
     'conditions Cloudy',        'conditions Fog',      'conditions Windy']
Length: 6, dtype: str
Road_traffic_density <ArrowStringArray>
['High ', 'Jam ', 'Low ', 'Medium ', 'NaN ']
Length: 5, dtype: str
City <ArrowStringArray>
['Urban ', 'Metropolitian ', 'Semi-Urban ', 'NaN ']
Length: 4, dtype: str
Time_taken(min) <ArrowStringArray>
['(min) 24', '(min) 33', '(min) 26', '(min) 21', '(min) 30', '(min) 40']
Length: 6, dtype: str


In [4]:
print('dates:', raw.Order_Date.min(), '→', raw.Order_Date.max())
print('zero/negative restaurant latitude:', (raw.Restaurant_latitude <= 0).sum())
print('ratings > 5:', (pd.to_numeric(raw.Delivery_person_Ratings, errors='coerce') > 5).sum())

dates: 01-03-2022 → 31-03-2022
zero/negative restaurant latitude: 4071
ratings > 5: 53


## 2. Merge the two weather exports
The Feb–Jul 2022 export is in **GMT** and has Chennai/Kolkata `location_id`s swapped relative to the Jul 2022→2026 **IST** export. Cities are matched by coordinates, GMT is shifted +5:30, and the IST export wins on the 1–2 Jul overlap.

In [5]:
mw.main()
weather = pd.read_csv(mw.OUT)
weather.head()

Wrote 229,650 rows -> C:\Users\Pratham\Downloads\college proj\ml\data\processed\weather_merged.csv
                          min                 max  count
city                                                    
Bangalore 2022-02-19 05:00:00 2026-07-02 23:00:00  38275
Chennai   2022-02-19 05:00:00 2026-07-02 23:00:00  38275
Delhi     2022-02-19 05:00:00 2026-07-02 23:00:00  38275
Kolkata   2022-02-19 05:00:00 2026-07-02 23:00:00  38275
Mumbai    2022-02-19 05:00:00 2026-07-02 23:00:00  38275
Pune      2022-02-19 05:00:00 2026-07-02 23:00:00  38275


,city,latitude,longitude,datetime_ist,date,hour,temp_c,precip_mm_day,rain_mm_day,temp_max_c,temp_min_c,wind_max_kmh,wind_gust_max_kmh,source
0,Bangalore,12.970123,77.56364,2022-02-19 05:00:00,2022-02-19,5,16.4,0.0,0.0,29.7,15.9,15.9,NaN,feb_jul_2022
1,Bangalore,12.970123,77.56364,2022-02-19 06:00:00,2022-02-19,6,16.0,0.0,0.0,29.7,15.9,15.9,NaN,feb_jul_2022
2,Bangalore,12.970123,77.56364,2022-02-19 07:00:00,2022-02-19,7,16.9,0.0,0.0,29.7,15.9,15.9,NaN,feb_jul_2022
3,Bangalore,12.970123,77.56364,2022-02-19 08:00:00,2022-02-19,8,19.5,0.0,0.0,29.7,15.9,15.9,NaN,feb_jul_2022
4,Bangalore,12.970123,77.56364,2022-02-19 09:00:00,2022-02-19,9,22.5,0.0,0.0,29.7,15.9,15.9,NaN,feb_jul_2022


## 3. Clean orders
- strip whitespace, `'NaN'` → missing
- remove `conditions ` / `(min) ` prefixes
- `Metropolitian` → `Metropolitan`
- city name from rider-ID prefix (`BANGRES…` → Bangalore)
- abs() on negative coordinates, ~0 coordinates → missing
- ratings > 5 → missing
- features: haversine distance, prep time (midnight-safe), hour, weekday, weekend, peak, holiday

In [6]:
df = co.clean_orders(raw)
for c in ['Weatherconditions', 'Road_traffic_density', 'City', 'Time_taken(min)']:
    print(c, df[c].unique()[:6])
df.city_name.value_counts()

Weatherconditions <ArrowStringArray>
['Sunny', 'Stormy', 'Sandstorms', 'Cloudy', 'Fog', 'Windy']
Length: 6, dtype: string
Road_traffic_density <ArrowStringArray>
['High', 'Jam', 'Low', 'Medium', <NA>]
Length: 5, dtype: string
City <ArrowStringArray>
['Urban', 'Metropolitan', 'Semi-Urban', <NA>]
Length: 4, dtype: string
Time_taken(min) [24 33 26 21 30 40]


city_name
Jaipur        3443
Ranchi        3229
Bangalore     3195
Surat         3187
Hyderabad     3181
Mumbai        3173
Mysore        3171
Coimbatore    3170
Vadodara      3166
Indore        3159
Chennai       3145
Pune          3132
Agra           763
Ludhiana       758
Kanpur         740
Allahabad      740
Dehradun       737
Goa            709
Aurangabad     703
Kochi          701
Kolkata        700
Bhopal         691
Name: count, dtype: int64

## 4. Keep orders from the first weather date (19 Feb 2022)
The orders have no dates between 19 and 28 Feb, so this drops exactly the 11–18 Feb orders. All Kolkata orders fall in that week, so Kolkata leaves the weather subset.

In [7]:
before = len(df)
df = df[df.Order_Date >= co.WEATHER_START]
print(f'{before:,} → {len(df):,} orders')
df.Order_Date.agg(['min', 'max'])

45,593 → 38,351 orders


min   2022-03-01
max   2022-04-06
Name: Order_Date, dtype: datetime64[us]

## 5. Join weather on (city, date, hour) and impute

In [8]:
df = co.join_weather(df, weather)
df['has_weather'] = df.temp_c.notna().astype(int)
print(df.groupby('city_name').has_weather.mean().sort_values(ascending=False).head(6))
df = co.impute(df)

city_name
Bangalore     1.0
Chennai       1.0
Pune          1.0
Mumbai        1.0
Hyderabad     0.0
Coimbatore    0.0
Name: has_weather, dtype: float64


In [9]:
model = df[df.has_weather == 1]
print('all cleaned orders:', len(df), '| with weather:', len(model))
model.city_name.value_counts()

all cleaned orders: 38351 | with weather: 12645


city_name
Bangalore    3195
Mumbai       3173
Chennai      3145
Pune         3132
Name: count, dtype: int64

In [10]:
cols = ['Time_taken(min)', 'distance_km', 'prep_time_min', 'temp_c', 'rain_mm_day',
        'wind_max_kmh', 'Delivery_person_Age', 'Delivery_person_Ratings']
model[cols].describe().round(2)

,Time_taken(min),distance_km,prep_time_min,temp_c,rain_mm_day,wind_max_kmh,Delivery_person_Age,Delivery_person_Ratings
count,12645.0,12645.00,12645.00,12645.00,12645.0,12645.00,12645.0,12645.0
mean,26.2,9.65,10.02,28.03,0.1,19.45,29.51,4.63
std,9.3,5.55,4.01,3.28,0.5,4.16,5.68,0.34
min,10.0,1.53,5.00,18.80,0.0,10.60,15.0,1.0
25%,19.0,4.66,5.00,25.90,0.0,16.20,25.0,4.6
50%,25.0,9.19,10.00,27.70,0.0,19.40,30.0,4.7
75%,32.0,13.77,15.00,30.10,0.0,22.40,34.0,4.8
max,54.0,20.18,15.00,39.80,4.8,29.40,50.0,5.0


In [11]:
model.isna().sum()[lambda s: s > 0]  # Time_Orderd raw column & holiday_name are expected

Time_Orderd       507
holiday_name    11629
dtype: int64

## 6. Save

In [12]:
df.to_csv(co.OUT_ALL, index=False)
model.to_csv(co.OUT_MODEL, index=False)
print('saved', co.OUT_ALL.name, co.OUT_MODEL.name)

saved orders_clean.csv model_data.csv
